In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import copy

In [20]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    print('Using CPU')

Using GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load metadata
df = pd.read_csv(r"C:\SUTD\50.021 Artificial Intelligence\pppppp\nihfull\Data_Entry_2017.csv")

# Keep only filename, patient id, and finding labels
df = df[["Image Index", "Patient ID", "Finding Labels"]].copy()

# Binary label: 1 if Pneumonia is present, else 0
df["label"] = df["Finding Labels"].apply(lambda x: 1 if "Pneumonia" in x.split("|") else 0)

# Remove rows with missing values just in case
df = df.dropna(subset=["Image Index", "Patient ID", "label"])

print(df["label"].value_counts())

label
0    110689
1      1431
Name: count, dtype: int64


In [22]:
patient_df = (
    df.groupby("Patient ID")["label"]
    .max()
    .reset_index()
)

print(patient_df["label"].value_counts())

label
0    29797
1     1008
Name: count, dtype: int64


In [23]:
sample_frac = 0.30

pos_patients = patient_df[patient_df["label"] == 1]
neg_patients = patient_df[patient_df["label"] == 0]

pos_sampled = pos_patients.sample(frac=sample_frac, random_state=42)
neg_sampled = neg_patients.sample(frac=sample_frac, random_state=42)

sampled_patients = pd.concat([pos_sampled, neg_sampled])

In [24]:
df_small = df[df["Patient ID"].isin(sampled_patients["Patient ID"])].copy()

print("Total sampled images:", len(df_small))
print(df_small["label"].value_counts())

Total sampled images: 33405
label
0    32981
1      424
Name: count, dtype: int64


In [25]:
train_patients, temp_patients = train_test_split(
    sampled_patients,
    test_size=0.3,
    stratify=sampled_patients["label"],
    random_state=42
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.5,
    stratify=temp_patients["label"],
    random_state=42
)

In [26]:
train_df = df_small[df_small["Patient ID"].isin(train_patients["Patient ID"])].copy()
val_df   = df_small[df_small["Patient ID"].isin(val_patients["Patient ID"])].copy()
test_df  = df_small[df_small["Patient ID"].isin(test_patients["Patient ID"])].copy()

print("Train:", len(train_df), train_df["label"].value_counts().to_dict())
print("Val:  ", len(val_df), val_df["label"].value_counts().to_dict())
print("Test: ", len(test_df), test_df["label"].value_counts().to_dict())

Train: 23499 {0: 23198, 1: 301}
Val:   5015 {0: 4956, 1: 59}
Test:  4891 {0: 4827, 1: 64}


In [27]:
# train_df.to_csv("train_split.csv", index=False)
# val_df.to_csv("val_split.csv", index=False)
# test_df.to_csv("test_split.csv", index=False)

In [28]:
# import os
# import shutil
# import pandas as pd

# def make_imagefolder_split(csv_file, source_img_dir, output_split_dir):
#     df = pd.read_csv(csv_file)

#     for _, row in df.iterrows():
#         img_name = row["Image Index"]
#         label = row["label"]

#         class_name = "pneumonia" if label == 1 else "not_pneumonia"

#         src = os.path.join(source_img_dir, img_name)
#         dst_dir = os.path.join(output_split_dir, class_name)
#         dst = os.path.join(dst_dir, img_name)

#         os.makedirs(dst_dir, exist_ok=True)
#         shutil.copy2(src, dst)   # or use os.symlink if you want

In [29]:
# src = r"C:\SUTD\50.021 Artificial Intelligence\niheverything"
# make_imagefolder_split("train_split.csv", src, "dataset/train")
# make_imagefolder_split("val_split.csv", src, "dataset/val")
# make_imagefolder_split("test_split.csv", src, "dataset/test")

In [30]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [31]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

class ChestXrayDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_name = row["Image Index"]
        label = int(row["label"])   # ensure int

        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [32]:
from torch.utils.data import DataLoader
src = r"C:\SUTD\50.021 Artificial Intelligence\niheverything"
train_dataset = ChestXrayDataset("train_split.csv", src, transform=train_transform)
val_dataset = ChestXrayDataset("val_split.csv", src, transform=val_transform)
test_dataset = ChestXrayDataset("test_split.csv", src, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [33]:
labels = [train_dataset[i][1] for i in range(1000)]  # sample first 1000
print(set(labels))

{0, 1}


In [34]:
# ChestXnet architecture

class CheXNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.densenet = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        self.densenet.classifier = nn.Sequential(
            nn.Linear(self.densenet.classifier.in_features, 1),
            nn.Sigmoid()    
        )
    
    def forward(self, x):
        x = self.densenet(x)
        return x

In [35]:
def downsample_train_negatives_to_ratio(train_df, neg_per_pos=10, seed=42):
    pos_df = train_df[train_df["label"] == 1]
    neg_df = train_df[train_df["label"] == 0]

    max_neg = min(len(neg_df), len(pos_df) * neg_per_pos)
    neg_sampled = neg_df.sample(n=max_neg, random_state=seed)

    train_balanced = (
        pd.concat([pos_df, neg_sampled])
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )
    return train_balanced

In [36]:
train_df = downsample_train_negatives_to_ratio(train_df, neg_per_pos=10)

In [37]:
print(train_df["label"].value_counts())
print(train_df["label"].value_counts(normalize=True))


label
0    3010
1     301
Name: count, dtype: int64
label
0    0.909091
1    0.090909
Name: proportion, dtype: float64


In [38]:
num_pos = (train_df["label"] == 1).sum()
num_neg = (train_df["label"] == 0).sum()
pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(device)

In [40]:
print("Train:", len(train_df), train_df["label"].value_counts().to_dict())
print("Val:  ", len(val_df), val_df["label"].value_counts().to_dict())
print("Test: ", len(test_df), test_df["label"].value_counts().to_dict())

Train: 3311 {0: 3010, 1: 301}
Val:   5015 {0: 4956, 1: 59}
Test:  4891 {0: 4827, 1: 64}


In [39]:
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

model = CheXNet().to(device)


for param in model.parameters():
    param.requires_grad = False
for param in model.densenet.classifier.parameters():
    param.requires_grad = True

optimizer = Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-5
)

num_pos = (train_df["label"] == 1).sum()
num_neg = (train_df["label"] == 0).sum()
pos_weight = torch.tensor([10.0]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)     
scheduler = ReduceLROnPlateau(optimizer, patience=1, factor=0.1)

scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())



def train_epoch(loader):
    total_loss = 0
    model.train()
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float().unsqueeze(1)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float().unsqueeze(1)

            logits = model(imgs)
            total_loss += criterion(logits, labels).item()
            all_preds.append(torch.sigmoid(logits).cpu())
            all_labels.append(labels.cpu())

    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    binary_preds = (preds > 0.5).float()
    accuracy = accuracy_score(labels.numpy().ravel(), binary_preds.numpy().ravel())

    return accuracy, total_loss / len(loader), preds, labels



best_val_loss = float("inf")
prev_lr = optimizer.param_groups[0]["lr"]
num_decays = 0

for epoch in range(100):
    train_loss = train_epoch(train_loader)
    val_accuracy, val_loss, preds, labels = evaluate(val_loader)
    auc = roc_auc_score(labels.numpy(), preds.numpy())

    print(f"Epoch {epoch+1} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | AUC: {auc:.4f} | val_accuracy: {val_accuracy}")

    scheduler.step(val_loss)

    curr_lr = optimizer.param_groups[0]["lr"]              # fix 6
    if curr_lr < prev_lr:
        num_decays += 1
        print(f"LR decayed to {curr_lr:.2e} ({num_decays}/3)")
        prev_lr = curr_lr

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved best model")

    if num_decays >= 3:
        print("Third LR decay reached, stopping.")
        break

model.load_state_dict(torch.load("best_model.pth"))

test_accuracy, test_loss, preds, labels = evaluate(test_loader)
auc = roc_auc_score(labels.numpy(), preds.numpy())
print(f"Test accuracy: {test_accuracy:.4f} | test_loss: {test_loss:.4f} | AUC: {auc:.4f}")



Epoch 1 | train_loss: 0.7868 | val_loss: 0.7676 | AUC: 0.5434 | val_accuracy: 0.011764705882352941
Saved best model


KeyboardInterrupt: 